# Complete LLM Zoomcamp Module 1 Notes: RAG & Agents with Google Gemini

---

## Part 1: RAG Fundamentals

---

### 01 - Introduction to RAG

#### What are LLMs?
Large Language Models are neural networks trained on massive amounts of text. Given a prompt, they generate a plausible continuation. In this course, we treat LLMs as black boxes accessed via API—text goes in, text comes out.

#### LLM Limitations

| Limitation | Description |
|------------|-------------|
| **Knowledge Cutoff** | Only knows training data, not recent events |
| **No Access to Your Data** | Cannot see internal documents or databases |
| **Hallucinations** | Confidently generates incorrect information |

#### What is RAG?
**RAG = Retrieval-Augmented Generation**

```mermaid
flowchart LR
    User[User Question] --> Search[Search / Retrieval]
    Search --> Context[Retrieved Documents]
    Context --> Prompt[Augmented Prompt]
    Prompt --> Gemini[Gemini / Generation]
    Gemini --> Answer[Grounded Answer]
```

#### Basic RAG Architecture
```python
def rag(question):
    search_results = search(question)           # Retrieval
    user_prompt = build_prompt(question, search_results)  # Augmentation
    return llm(user_prompt)                     # Generation
```

**Three Core Components:**
1. **Search** - Finds relevant documents from knowledge base
2. **Prompt** - Combines question with retrieved context
3. **LLM** - Generates grounded answer (Gemini)

---

### 02 - Environment Setup with uv

#### Prerequisites
- Python 3.10 or later (3.11+ recommended for compatibility)
- Google AI Studio account (for Gemini API)
- Basic Python and command line familiarity

#### Install uv Package Manager

**Mac/Linux:**
```bash
curl -LsSf https://astral.sh/uv/install.sh | sh
```

**Windows:**
```powershell
powershell -ExecutionPolicy ByPass -c "irm https://astral.sh/uv/install.ps1 | iex"
```

#### Project Setup
```bash
mkdir llm-zoomcamp
cd llm-zoomcamp
uv init
```

#### Install Dependencies with Google Gemini
```bash
uv add requests minsearch google-generativeai jupyter python-dotenv
```

Or as dev dependencies:
```bash
uv add --dev google-generativeai python-dotenv
```

**What each package does:**
- `requests` - Fetch FAQ dataset from internet
- `minsearch` - In-memory search engine for indexing and searching
- `google-generativeai` - Google Gemini API client (legacy SDK)
- `jupyter` - Notebook environment
- `python-dotenv` - Load API keys from `.env` file

#### Sync Environment
```bash
uv sync
```

#### API Key Management

Create `.env` file:
```bash
GOOGLE_API_KEY=AIzaSy...your_full_key_here
```

Add to `.gitignore`:
```
.env
```

> **⚠️ CRITICAL:** Never commit `.env` to git. Treat API keys like passwords.

#### Verify `pyproject.toml`
```toml
dependencies = [
    "google-generativeai>=0.8.0",
    "python-dotenv>=1.0.0",
    "requests",
    "minsearch",
    "jupyter"
]
```

#### Start Jupyter
```bash
uv run jupyter notebook
```

#### Test Gemini Setup (Legacy SDK - Used Throughout This Course)
```python
import os
import google.generativeai as genai
from dotenv import load_dotenv

load_dotenv()
genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

def llm(prompt):
    model = genai.GenerativeModel('gemini-2.5-flash')
    response = model.generate_content(prompt)
    return response.text

# Test
result = llm("Explain quantum computing in simple terms")
print(result)
```

#### Test Gemini Setup (Newer SDK - Future Compatible)
```python
from google import genai

client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents="Explain quantum computing in simple terms"
)
print(response.text)
```

> **SDK Note:** Both versions work. This course uses the legacy `google.generativeai` SDK throughout because its `start_chat()` interface is currently more mature for multi-turn agent loops. However, Google is actively transitioning to the newer `from google import genai` SDK. Check [Google AI Studio documentation](https://aistudio.google.com/) for the latest SDK status and pricing.

#### Quick Verification
```bash
uv pip list | grep generativeai
```

---

### 03 - Understanding RAG in Depth

#### Why Plain LLMs Fail for Specific Questions

**Example Question:** "I just discovered the course. Can I join now?"

- Gemini alone → generic answer (doesn't know course policies)
- With context → correct, specific answer

#### Manual Context Addition
```python
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your
project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to
receive the confirmation email?
You don't need it. You're accepted. You can also just start learning
and submitting homework.
"""

prompt = """
Your task is to answer questions from the course participants based
on the provided context.

Use the context to find relevant information and provide accurate answers.
If the answer is not found in the context, respond with "I don't know."

Question: {question}
Context: {context}
"""

answer = llm(prompt)
```

#### RAG Flow Diagram
```mermaid
flowchart TB
    User[User Question] --> App[Application]
    App --> DB[(Knowledge Base)]
    DB --> Docs[Retrieved Documents]
    Docs --> Build[Build Prompt]
    Build --> Combined[Question + Context]
    Combined --> LLM[Gemini]
    LLM --> Answer[Grounded Answer]
```

#### Key Insight
> "Your model is only as good as your retrieval. Search quality matters a lot for RAG."

---

### 04 - The Course FAQ Dataset

#### Data Source
- FAQ data from DataTalks.Club courses
- URL: `https://datatalks.club/faq/json/courses.json`

#### Fetching the Data
```python
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"{url_prefix}{course['path']}"
    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()
    documents.extend(course_data)

print(f"Loaded {len(documents)} documents")  # ~1100 documents
```

#### Document Structure
```python
{
    "id": "0e38656cfb",
    "course": "machine-learning-zoomcamp",  # For filtering
    "section": "General Course-Related Questions",  # For ranking
    "question": "How do I submit homework?",  # Primary search field
    "answer": "- Do the tasks locally\n- Publish your code..."  # Secondary search
}
```

#### Field Purposes

| Field | Purpose |
|-------|---------|
| `id` | Unique identifier |
| `course` | Filtering (exact match) |
| `section` | Ranking context |
| `question` | Primary search text |
| `answer` | Secondary search text |

#### Data Preparation Note
In real-world RAG systems, data preparation is often the most time-consuming part:
- Web scraping
- PDF parsing
- Document cleaning
- Text chunking

> This dataset was pre-cleaned with LLM assistance for the course.

---

### 05 - Search with Minsearch

#### Search Basics
Every search engine does the same thing:
1. Takes a query
2. Scores every document for similarity
3. Returns top N results

```
score = sim(query, document)
```

#### Text Search vs Semantic Search

| Type | Method | Example |
|------|--------|---------|
| **Text/Lexical** | Counts shared words (exact matching) | "join" matches "join" only |
| **Vector/Semantic** | Compares meaning | "join course" matches "enroll late" |

**Example where text search fails:**
- Query: "Can I still join the course after the start date?"
- Document: "Is it possible to enroll late?"
- Same meaning, different keywords → no match

#### Why Search Matters
- ~1100 documents in our dataset
- Sending all to Gemini = expensive and slow
- Search finds the most relevant 5 documents instead

#### Minsearch Setup
```python
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],  # Tokenized and ranked
    keyword_fields=["course"]  # Exact string matching for filtering
)

index.fit(documents)
```

#### Text Fields vs Keyword Fields

| Field Type | Processing | Use Case |
|------------|------------|----------|
| **Text** | Tokenized, lowercased, stop words removed, ranked by relevance | Searchable content |
| **Keyword** | Exact string matching, no processing | Filtering, faceting |

#### Performing a Search
```python
question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)
```

#### Field Boosting
Not all fields are equally important. Default boost = 1.0.

```python
boost_dict = {
    "question": 2.0,  # Twice as important
    "section": 0.5    # Half as important
}
```

#### Filtering by Course
```python
# Only MLOps Zoomcamp results
results = index.search(
    question,
    filter_dict={"course": "mlops-zoomcamp"}
)
```

#### Wrapped Search Function
```python
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}
    
    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )
```

---

### Complete RAG Pipeline with Google Gemini

#### Full Implementation
```python
import os
import requests
import google.generativeai as genai
from minsearch import Index
from dotenv import load_dotenv

# 1. Setup
load_dotenv()
genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

# 2. Load and index documents
docs_url = "https://datatalks.club/faq/json/courses.json"
courses_raw = requests.get(docs_url).json()

documents = []
url_prefix = "https://datatalks.club/faq"
for course in courses_raw:
    course_url = f"{url_prefix}{course['path']}"
    course_data = requests.get(course_url).json()
    documents.extend(course_data)

# 3. Create search index
index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)
index.fit(documents)

# 4. Search function
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}
    
    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

# 5. Gemini LLM function
def llm(prompt):
    model = genai.GenerativeModel('gemini-1.5-flash')
    response = model.generate_content(prompt)
    return response.text

# 6. Build prompt function
def build_prompt(question, search_results):
    context = "\n\n".join([
        f"Q: {doc['question']}\nA: {doc['answer']}"
        for doc in search_results
    ])
    
    return f"""
Your task is to answer questions from course participants based on the provided context.

Use the context to find relevant information and provide accurate answers.
If the answer is not found in the context, respond with "I don't know."

Question: {question}

Context:
{context}
"""

# 7. Complete RAG pipeline
def rag(question, course="llm-zoomcamp"):
    search_results = search(question, course)
    prompt = build_prompt(question, search_results)
    return llm(prompt)

# 8. Usage
answer = rag("I just discovered the course. Can I join now?")
print(answer)
```

#### Available Gemini Models

> **Model Selection Note:** Use the latest generally available Gemini Flash model for most use cases unless your task specifically requires Pro-level reasoning. Check [Google AI Studio](https://aistudio.google.com/) for current model availability.

| Model Type | Description | Best For |
|------------|-------------|----------|
| **Gemini Flash** | Fast, efficient, cost-effective | Most RAG use cases (recommended) |
| **Gemini Pro** | More capable, deeper reasoning | Complex multi-step tasks |
| **Gemini Experimental** | Latest features | Testing new capabilities |

#### Streaming Responses
```python
def llm_stream(prompt):
    model = genai.GenerativeModel('gemini-1.5-flash')
    response = model.generate_content(prompt, stream=True)
    
    for chunk in response:
        print(chunk.text, end='')
```

---

### 06 - Building the Prompt

#### Why Split the Prompt?
When building AI systems, split the prompt into two parts:

| Part | Description | Frequency |
|------|-------------|-----------|
| **Instructions (System Prompt)** | Tells LLM how to behave | Fixed, same for every request |
| **User Prompt** | Contains question + context | Changes with every request |

#### Instructions Template
```python
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""
```

#### User Prompt Template
```python
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""
```

#### Building Context from Search Results
```python
def build_context(search_results):
    lines = []
    
    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")
    
    return "\n".join(lines).strip()
```

#### Building Complete Prompt
```python
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()
```

#### Example Output
```text
Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework...
```

#### Key Insight
> "The prompt is the bridge between search and the LLM. A bad prompt lets the LLM ignore the context and hallucinate. A good prompt keeps the answer grounded."

---


### 07 - The LLM (Google Gemini)

#### Sending Prompt to Gemini

```python
import google.generativeai as genai

def llm(instructions, user_prompt, model="gemini-1.5-flash"):
    # Combine instructions and user prompt
    full_prompt = f"{instructions}\n\n{user_prompt}"
    
    model_instance = genai.GenerativeModel(model)
    response = model_instance.generate_content(full_prompt)
    
    # Safety check - handle blocked responses
    if not response.candidates:
        return "Response blocked by safety filters."
    
    return response.text
```

#### Message History Alternative (Chat Mode)
```python
def llm_with_history(instructions, user_prompt, model="gemini-1.5-flash"):
    model_instance = genai.GenerativeModel(model)
    
    chat = model_instance.start_chat(history=[])
    
    # Send instructions as system prompt
    chat.send_message(instructions)
    
    # Send user prompt
    response = chat.send_message(user_prompt)
    
    return response.text
```

#### Token Usage with Gemini
```python
def llm_with_usage(instructions, user_prompt, model="gemini-1.5-flash"):
    model_instance = genai.GenerativeModel(model)
    full_prompt = f"{instructions}\n\n{user_prompt}"
    
    response = model_instance.generate_content(full_prompt)
    
    # Gemini usage metadata
    print(f"Prompt tokens: {response.usage_metadata.prompt_token_count}")
    print(f"Candidate tokens: {response.usage_metadata.candidates_token_count}")
    print(f"Total tokens: {response.usage_metadata.total_token_count}")
    
    return response.text
```

#### Cost Calculation for Gemini

> **⚠️ Important:** Check current Gemini pricing in [Google AI Studio documentation](https://aistudio.google.com/pricing). Prices change frequently. The example below is illustrative only.

```python
def calculate_gemini_cost(usage_metadata):
    # ILLUSTRATIVE ONLY - Verify current rates at aistudio.google.com/pricing
    INPUT_PRICE_PER_MILLION = 0.075   # Example rate
    OUTPUT_PRICE_PER_MILLION = 0.30   # Example rate
    
    cost = (
        usage_metadata.prompt_token_count * (INPUT_PRICE_PER_MILLION / 1_000_000) +
        usage_metadata.candidates_token_count * (OUTPUT_PRICE_PER_MILLION / 1_000_000)
    )
    
    return cost

# Usage
response = model.generate_content(prompt)
cost = calculate_gemini_cost(response.usage_metadata)
print(f"Estimated cost: ${cost:.6f}")
```

#### Complete LLM Function for RAG
```python
def llm(instructions, user_prompt, model="gemini-1.5-flash"):
    full_prompt = f"{instructions}\n\n{user_prompt}"
    
    model_instance = genai.GenerativeModel(model)
    response = model_instance.generate_content(full_prompt)
    
    return response.text
```

#### Full RAG Pipeline with Gemini
```python
def rag(query, model="gemini-1.5-flash"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

# Test
answer = rag("I just discovered the course. Can I join now?")
print(answer)
```

#### Modular Design Benefits
- ✅ Swap search backend without changing LLM code
- ✅ Swap prompt template without changing search or LLM
- ✅ Change Gemini model version with one parameter

---


### 08 - RAG Helper (Refactored)

Create reusable files to avoid code duplication.

#### `ingest.py` - Data Loading and Indexing
```python
import requests
from minsearch import Index

def load_faq_data():
    docs_url = "https://datatalks.club/faq/json/courses.json"
    response = requests.get(docs_url)
    courses_raw = response.json()
    
    documents = []
    url_prefix = "https://datatalks.club/faq"
    
    for course in courses_raw:
        course_url = f"{url_prefix}{course['path']}"
        course_response = requests.get(course_url)
        course_response.raise_for_status()
        course_data = course_response.json()
        documents.extend(course_data)
    
    return documents

def build_index(documents):
    index = Index(
        text_fields=["question", "section", "answer"],
        keyword_fields=["course"]
    )
    index.fit(documents)
    return index
```

#### `rag_helper.py` - RAG Logic (Gemini Version)
```python
import google.generativeai as genai

INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

PROMPT_TEMPLATE = """
QUESTION: {question}

CONTEXT:
{context}
""".strip()

class RAGBase:
    def __init__(
        self,
        index,
        instructions=INSTRUCTIONS,
        prompt_template=PROMPT_TEMPLATE,
        course="llm-zoomcamp",
        model="gemini-1.5-flash"
    ):
        self.index = index
        self.instructions = instructions
        self.course = course
        self.prompt_template = prompt_template
        self.model = model
    
    def search(self, query, num_results=5):
        boost_dict = {"question": 3.0, "section": 0.5}
        filter_dict = {"course": self.course}
        
        return self.index.search(
            query,
            num_results=num_results,
            boost_dict=boost_dict,
            filter_dict=filter_dict
        )
    
    def build_context(self, search_results):
        lines = []
        
        for doc in search_results:
            lines.append(doc["section"])
            lines.append("Q: " + doc["question"])
            lines.append("A: " + doc["answer"])
            lines.append("")
        
        return "\n".join(lines).strip()
    
    def build_prompt(self, query, search_results):
        context = self.build_context(search_results)
        return self.prompt_template.format(
            question=query, context=context
        )
    
    def llm(self, prompt):
        full_prompt = f"{self.instructions}\n\n{prompt}"
        model_instance = genai.GenerativeModel(self.model)
        response = model_instance.generate_content(full_prompt)
        return response.text
    
    def rag(self, query):
        search_results = self.search(query)
        prompt = self.build_prompt(query, search_results)
        answer = self.llm(prompt)
        return answer
```

#### Using the Helper in a Notebook
```python
from dotenv import load_dotenv
load_dotenv()

import os
import google.generativeai as genai
from ingest import load_faq_data, build_index
from rag_helper import RAGBase

# Configure Gemini
genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

# Load data and build index
documents = load_faq_data()
index = build_index(documents)

# Create assistant
assistant = RAGBase(
    index=index,
)

# Test
answer = assistant.rag("I just discovered the course. Can I join now?")
print(answer)
```

#### Custom Instructions Example
```python
custom_instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
Be concise and helpful.
""".strip()

assistant = RAGBase(
    index=index,
    instructions=custom_instructions,
    model="gemini-1.5-pro"  # Use more capable model for complex queries
)
```

---


### 09 - Data Ingestion (Persistent Storage)

#### Why Separate Ingestion from Querying?

**Problem with Minsearch:**
- In-memory only - data disappears when process stops
- Re-indexes on every restart (wasteful for large datasets)
- Cannot have separate write/read processes

**Solution with SQLite:**
- Persistent storage on disk
- Index once, query many times
- Allows concurrent reads during writes

#### Install SQLiteSearch
```bash
uv add sqlitesearch
```

#### Ingestion Notebook (`sqlite-ingest.ipynb`)
```python
import time
from sqlitesearch import TextSearchIndex
from ingest import load_faq_data

# Load data
documents = load_faq_data()
print(f"Loaded {len(documents)} documents")

# Filter to specific course
docs_llm = [doc for doc in documents if doc["course"] == "llm-zoomcamp"]
print(f"LLM Zoomcamp: {len(docs_llm)} documents")

# Create persistent index
index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq.db"  # Persists to disk
)

# Add documents (simulate slow ingestion)
for doc in docs_llm:
    index.add(doc)
    print(f"Added: {doc['question'][:60]}...")
    time.sleep(0.5)

index.close()
print("Done. Index saved to faq.db")
```

#### Query Notebook (`persistent_rag.ipynb`)
```python
from sqlitesearch import TextSearchIndex
from rag_helper import RAGBase
import google.generativeai as genai
from dotenv import load_dotenv

load_dotenv()
genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

# Connect to existing index
sqlite_index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq.db"
)

# Check document count
print(f"Documents in index: {sqlite_index.count()}")

# Search while ingestion runs
results = sqlite_index.search(
    "Can I still join the course after it started?",
    num_results=5
)

# Use with RAG helper
assistant = RAGBase(
    index=sqlite_index,
)

answer = assistant.rag("Can I still join the course after it started?")
print(answer)

# Clean up
sqlite_index.close()
```

#### Architecture Comparison

**Minsearch (Single Process):**
```
Startup: fetch data → parse → index → ready
Every restart: repeat all steps
```

**SQLiteSearch (Two Processes):**
```
Ingestion (runs once): fetch data → parse → write to faq.db
Query (runs every time): open faq.db → search → ready
```

#### Architecture Diagram
```mermaid
flowchart TB
    subgraph Ingestion["Ingestion Process (Runs Once)"]
        FAQ[FAQ.json] --> Parser[Ingestor - Parse & Index]
        Parser --> DB[(faq.db)]
    end
    
    subgraph Query["Query Process (Runs Every Time)"]
        User[User Question] --> Read[Read DB]
        Read --> Search[Search]
        Search --> Gemini[Gemini]
        Gemini --> Answer[Answer]
    end
    
    DB -.-> Read
```

#### Choosing the Right Tool

| Tool | Use Case | Pros | Cons |
|------|----------|------|------|
| **Minsearch** | Small data, fast indexing | Simple, no dependencies | In-memory only |
| **SQLiteSearch** | Large data, persistence | File-based, concurrent reads | Slower than Elasticsearch |
| **Elasticsearch** | Production scale | Distributed, real-time | Heavy, requires Docker |
| **Qdrant/Weaviate** | Vector search | Hybrid search | Specialized |

#### Transition to Module 2: Vector Search Motivation

**Keyword Search (Current - Module 1):**
```
Query: "join course"
Must match: "join course" exactly
Cannot match: "enroll late" (different words, same meaning)
```

**Vector Search (Coming - Module 2):**
```mermaid
flowchart LR
    Text["join course"] --> Model[Embedding Model]
    Model --> Vector[Vector Representation]
    Vector --> Match[Semantic Match]
    Match --> Results["enroll late", "sign up after deadline", "join course"]
```

This is why Module 2 covers embeddings and vector search - to handle synonyms, paraphrases, and semantic similarity that keyword search misses.

---








### 10 - Wrap-up of Part 1

#### What We Accomplished
1. ✅ Understood RAG: Retrieval + Augmentation + Generation
2. ✅ Built search engine with minsearch over real FAQ data
3. ✅ Created prompt templates combining questions with context
4. ✅ Integrated Google Gemini as the LLM with safety handling
5. ✅ Wired everything into working RAG pipeline
6. ✅ Split ingestion and query with sqlitesearch for persistence

#### Two Directions Forward

**Direction 1: Agents (Brief Introduction Below)**
- LLM decides what to search for
- Can run multiple searches
- Handles typos and unusual phrasings
- Recovers from poor search results

**Direction 2: Vector Search (Module 2)**
- Semantic matching instead of keyword matching
- Handles synonyms and paraphrases
- Uses embeddings to capture meaning
- Example: "Can I still join?" matches "Is it possible to enroll late?"

#### Fine-tuning vs RAG

| Aspect | Fine-tuning | RAG |
|--------|-------------|-----|
| Hardware | Requires GPUs | Works with any LLM |
| Updates | Hard to update | Easy to add new data |
| Model | Changes model weights | Keeps model unchanged |
| Need | Rarely needed | Standard approach |

> "RAG is more flexible, cheaper, and works with any LLM. In practice, fine-tuning is rarely needed."

#### Self-Check: What You Should Know After Module 1

After completing Module 1, you should be able to:
- ✅ Explain RAG and its three components
- ✅ Build a minsearch index from FAQ data
- ✅ Retrieve relevant documents for a query
- ✅ Construct effective prompts with context
- ✅ Call Gemini APIs for grounded generation
- ✅ Build a complete RAG pipeline end-to-end
- ✅ Persist data using SQLite for production use
- ✅ Explain why vector search is needed (Module 2)

#### Next Steps to Try
1. Experiment with different prompt templates
2. Add more data sources to knowledge base
3. Try different Gemini models (Flash vs Pro)
4. Implement Elasticsearch as search backend

#### Complete Production-Ready RAG with Gemini
```python
import os
import google.generativeai as genai
from sqlitesearch import TextSearchIndex
from rag_helper import RAGBase
from dotenv import load_dotenv

load_dotenv()
genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

# Load persistent index
index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq.db"
)

# Create RAG assistant
assistant = RAGBase(
    index=index,
    model="gemini-1.5-flash"
)

# Interactive query loop
while True:
    question = input("\nAsk a question (or 'quit'): ")
    if question.lower() == 'quit':
        break
    answer = assistant.rag(question)
    print(f"\nAnswer: {answer}\n")
```

#### Key Takeaways for Production
1. **Separate ingestion from querying** - Index once, query many times
2. **Use persistent storage** - SQLite for small/medium, Elasticsearch for large
3. **Modular design** - Can swap search, LLM, or prompts independently
4. **Start with RAG** - Fine-tuning is rarely necessary
5. **Monitor costs** - Check current Gemini pricing regularly
6. **Ground answers** - Always provide context to reduce hallucinations
7. **Handle safety blocks** - Check for blocked responses in production code

---

## Part 2: Agents (Brief Introduction)

### 11 - Agents Introduction

#### The Problem with Fixed RAG Pipelines

Every RAG pipeline from Part 1 follows the same flow:
1. Search the FAQ
2. Build a prompt with results
3. Send to LLM

**This fails when:**
- User makes a typo (e.g., "Olama" instead of "Ollama")
- Question is phrased unusually
- Information is spread across multiple searches
- Search returns nothing useful (lexical search requires exact matches)

```
┌──────────────┐     ┌──────────────┐     ┌──────────────┐
│ User:        │     │ search -     │     │ LLM: I don't │
│ "How do I    │────►│ "Olama" -    │────►│ have info    │
│ run Olama?"  │     │ no results   │     │ about Olama" │
└──────────────┘     └──────────────┘     └──────────────┘
```

#### The Agent Alternative

An **agent** puts the LLM in control. Instead of a fixed flow, the LLM decides:
- Which actions to take
- In which order
- When to stop and answer

**With an agent:**
```
┌──────────────┐     ┌──────────────┐     ┌──────────────┐
│ User:        │     │ LLM: I'll    │     │ search -     │
│ "How do I    │────►│ search for   │────►│ "Olama" -    │
│ run Olama?"  │     │ 'Olama'      │     │ no results   │
└──────────────┘     └──────────────┘     └──────────────┘
                                                    │
                                                    ▼
┌──────────────┐     ┌──────────────┐     ┌──────────────┐
│ LLM: Here's  │     │ search -     │     │ LLM: Hmm,    │
│ how to run   │◄────│ "Ollama" -   │◄────│ maybe typo   │
│ Ollama...    │     │ found!       │     │ for Ollama"  │
└──────────────┘     └──────────────┘     └──────────────┘
```

#### Key Difference: RAG vs Agent

| Aspect | RAG (Fixed) | Agent (Flexible) |
|--------|-------------|------------------|
| **Control** | Developer decides steps | LLM decides actions |
| **Search count** | Fixed (1) | Variable (multiple) |
| **Typo handling** | None | Automatic correction |
| **Clarification** | Cannot ask questions | Can ask user for more info |
| **Recovery** | No recovery from bad search | Can retry with corrected terms |

---

### 12 - Agent Concepts: What's Under the Hood

> **Note:** A full treatment of agents (function calling, agentic loops, frameworks) is covered in depth in later modules. This section provides the conceptual foundation you need.

#### The Three Parts of an Agent

| Part | Description | Example |
|------|-------------|---------|
| **Instructions** | Role and behavior (system prompt) | "You are a course assistant. Search before answering." |
| **Tools** | Functions the agent can call | `search(query)` |
| **Memory** | Message history (what's been tried) | Full conversation log |

#### Why Understanding the Manual Loop Matters

While modern SDKs like Gemini's `enable_automatic_function_calling` handle the agent loop for you, understanding the manual pattern is crucial because **every agent framework (LangChain, PydanticAI, Google ADK) is wrapping this exact pattern:**

```python
# Conceptual agent loop (what frameworks do under the hood)
while not done:
    response = llm(messages)           # Ask LLM what to do
    if response is function_call:      # LLM wants to use a tool
        result = execute_tool(response) # Run the tool
        messages.append(result)         # Add result to history
    else:
        return response.text            # LLM is ready to answer
```

#### The Costs of Agents

Agents are powerful but come with trade-offs:

| Cost | Description |
|------|-------------|
| **Higher Latency** | Multiple round-trips to the LLM per question |
| **Higher Cost** | Every iteration bills for the full message history |
| **Unpredictability** | LLM might take different paths for the same prompt |
| **Complexity** | More moving parts to debug and monitor |

> **Rule of thumb:** Try the simpler approach first (plain RAG). If it works, ship it. Reach for agents only when simpler solutions genuinely can't handle the problem.

#### The Agent Framework Landscape

Key frameworks you'll encounter in later modules:

| Framework | Best For |
|-----------|----------|
| **Google ADK** | Native Gemini integration, Google Cloud users |
| **PydanticAI** | Type-safe, multi-provider support |
| **LangGraph** | Complex multi-step workflows |
| **ToyAIKit** | Learning and teaching agent concepts |

---

### Quick Start: Agentic RAG with Gemini

For those ready to experiment, here's the simplest approach using Gemini's built-in automatic function calling:

```python
import google.generativeai as genai

# Define search tool with type hints and docstring
def search(query: str) -> list:
    """Search the FAQ database for entries matching the given query."""
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

# Create model with tool - Gemini auto-generates the schema from your docstring
model = genai.GenerativeModel(
    'gemini-1.5-flash',
    tools=[search],  # Pass function directly
    system_instruction="You are a course assistant. Use search to find answers."
)

# Automatic function calling handles the loop for you
chat = model.start_chat(enable_automatic_function_calling=True)
response = chat.send_message("How do I run Olama locally?")
print(response.text)  # Corrects "Olama" to "Ollama" automatically
```

---

## Summary: Module 1 Architecture

```mermaid
flowchart TB
    subgraph RAG["RAG Pipeline (Module 1)"]
        direction LR
        Search[Search Engine] --> Prompt[Prompt Builder]
        Prompt --> LLM[Gemini LLM]
        LLM --> Answer[Grounded Answer]
    end
    
    subgraph Data["Data Layer"]
        Ingest[Ingestion] --> DB[(Persistent Storage)]
        DB --> Search
    end
    
    subgraph AgentPreview["Agent Preview (Module 4 Deep Dive)"]
        Tools[Tools / Functions] --> Loop[Agentic Loop]
        Loop --> Multi[Multi-Search Capability]
    end
    
    User[User Question] --> Search
```

---

**Course Progress:** Module 1 Complete ✓

**Key Takeaway:** RAG grounds LLM responses in your data through search and context. Start with simple RAG, add persistence for production, and only add agents when simple approaches fail.

**Next Module:** Vector Search with Embeddings - Semantic matching for better retrieval

---

## Appendix: Recommended Repository Structure

```
llm-zoomcamp-gemini/
├── README.md
├── notes/
│   ├── module_01_rag.md            # This document
│   ├── module_02_embeddings.md     # Coming next
│   ├── module_03_evaluation.md
│   └── module_04_agents.md
├── notebooks/
│   ├── module1_rag_basics.ipynb
│   ├── module1_persistent_rag.ipynb
│   └── module1_agent_preview.ipynb
├── code/
│   ├── ingest.py
│   ├── rag_helper.py
│   └── app.py
└── diagrams/
    └── architecture.mermaid
```